# 모듈 (4/5): 에이전트 메모리 · Deep-Knowledge Agent
## 지식(Memory & Knowledge) 시리즈 — 메모리를 직접 만들어 RAG 와 잇는 편

---

에이전트의 **단기(슬라이딩 윈도우) + 에피소딕 + 시맨틱** 메모리를 결합하고, 이를 Hybrid RAG 와
합쳐 **Multi-hop 추론이 가능한 Deep-Knowledge Agent** 를 완성합니다.

### 이 시리즈 구성
1. `M04_1_embeddings.ipynb` — 임베딩 · 시맨틱 유사도 · 공급자 비교
2. `M04_2_vector_rag.ipynb` — 문서 전처리 · 벡터 DB 3종 · LangChain RAG
3. `M04_3_graph_rag.ipynb` — 지식 그래프 · Neo4j · LlamaIndex GraphRAG
4. **`M04_4_agent_memory.ipynb`** ← (현재) 에이전트 메모리 **직접 구현** · Deep-Knowledge Agent
5. `M04_5_memory.ipynb` — LangGraph **메모리 인프라**(체크포인터 · Store · 영속화)

### 4편(이 노트북)과 5편은 무엇이 다른가

두 노트북 모두 "에이전트 메모리"를 다루지만 **목적과 구현 층이 다릅니다.**

| | **4편 `M04_4_agent_memory`** ← 현재 | **5편 [`M04_5_memory`](M04_5_memory.ipynb)** |
|---|---|---|
| 목적 | RAG 스택 위에 메모리를 얹어 **Deep-Knowledge Agent 완성** | LangGraph/LangChain 의 **표준 메모리 인프라** 학습 |
| 분류 축 | 인지 구조 — 단기 / 에피소딕 / 시맨틱 / 작업 | 저장 수명 — 단기 / 장기(영속) / 크로스 스레드 / 시맨틱 |
| 구현 방식 | 계층을 **직접 설계** 한 순수 파이썬 구현 ([`agentic_lib/agent_memory.py`](agentic_lib/agent_memory.py)) | 프레임워크 부품을 감싼 래퍼 ([`agentic_lib/memory_advanced.py`](agentic_lib/memory_advanced.py)) |
| 스택 | ChromaDB + 순수 파이썬 — **프레임워크 비의존** | `MessagesState` · `MemorySaver` · `SqliteSaver` · `InMemoryStore` |
| 결합 대상 | Vector / Graph / Hybrid RAG, Multi-hop 추론 | 대화 세션(`thread_id`) · 사용자 프로필 |
| 결과물 | `DeepKnowledgeAgent` | `FullMemoryAgent` + 메모리 전략 선택 가이드(비용 vs 성능) |

- **5편에만 있는 것** — 영속화(`SqliteSaver`: 프로세스를 재시작해도 기억), 스레드별 세션 분리,
  대화 히스토리 타임라인 조회(`get_state_history`), 크로스 스레드 사용자 프로필(LangGraph Store), 요약 메모리.
- **4편에만 있는 것** — 에피소딕 메모리(중요도·태그로 사건 저장), 작업 메모리,
  그리고 **RAG 와의 결합** — 메모리가 multi-hop 추론의 입력이 되는 부분.

> 📖 **권장 순서는 4편 → 5편** 입니다. 여기서 메모리를 직접 만들어 "왜 그렇게 동작하는가"를 이해한 뒤,
> 5편에서 같은 개념을 프레임워크가 어떤 부품으로 제공하는지("실무에서 무엇을 쓰는가") 봅니다.

### 학습 목표
1. 단기/에피소딕/시맨틱/작업 메모리를 결합한 `AgentMemorySystem`
2. Hybrid RAG + 메모리 + 추론 체인을 묶은 `DeepKnowledgeAgent`

> 📦 **환경 설치·실행 명령**은 [`env_guides/M04_4_agent_memory.md`](env_guides/M04_4_agent_memory.md) 참고.
> 이 노트북은 **자기완결**입니다 — 에이전트가 쓰는 Vector RAG · 지식 그래프 · Hybrid RAG 를
> 아래에서 먼저 재구성합니다.

In [1]:
# 자기완결 setup: 이 노트북만 단독 실행해도 되도록 공통 셋업을 맨 앞에 둔다
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
from typing import List, Dict, Optional, Any
from datetime import datetime
utils.reload_env()  # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import (
    uv_install, get_llm, test_llm_connection,
    LLM_PROVIDER, chunk_text, cosine_similarity,
)
# 공통 라이브러리: bootstrap(응답 정규화) / rag(RAG·지식그래프) / memory(메모리 프리미티브)
from agentic_lib import bootstrap, rag, memory
from agentic_lib import agent_memory as am   # 4계층 메모리 + DeepKnowledgeAgent(6~7절)
from agentic_lib.bootstrap import to_text    # 공급자 무관 응답 정규화(<think>·list content 제거)

# 이 노트북에 필요한 패키지만 보강 설치 (이미 있으면 건너뜀)
uv_install(['chromadb', 'sentence-transformers', 'numpy'])

llm = test_llm_connection()  # 공급자 무관 — None 이면 각 구현이 폴백 경로로 동작

LLM 공급자: openrouter
  OpenRouter Key: 설정됨  /  Model: nvidia/nemotron-3-super-120b-a12b:free
[uv] 설치 완료: ['chromadb', 'sentence-transformers', 'numpy']
LLM 연결 성공 [openrouter]: 1+1을 계산하면 2입니다.


---
### 선행 구축 (Vector RAG + 지식 그래프 + Hybrid RAG 재구성)

`DeepKnowledgeAgent` 는 **Hybrid RAG**(Vector + Graph)와 **메모리 시스템** 을 함께 사용합니다.
이 노트북을 단독 실행할 수 있도록, [2편](M04_2_vector_rag.ipynb)·[3편](M04_3_graph_rag.ipynb)의
ChromaDB 컬렉션 · 샘플 문서 · `VectorRAG` · `KnowledgeGraph` · `HybridRAG` 를 아래에서 먼저 재구성합니다.

In [2]:
# ChromaDB: 임베딩 벡터를 인덱싱/검색하는 로컬 벡터 데이터베이스
import chromadb
from chromadb.utils import embedding_functions

# ChromaDB 초기화 (인메모리 — 테스트용. 영구 저장은 PersistentClient 사용)
chroma_client = chromadb.Client()
# chroma_client = chromadb.PersistentClient(path="./chroma_db")  # 영구 저장

# 임베딩 함수: 위와 동일한 오프라인 다국어 모델 사용
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

# 컬렉션 생성 (코사인 거리 기준 HNSW 인덱스)
collection = chroma_client.get_or_create_collection(
    name="agentic_ai_docs",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB 컬렉션 생성: {collection.name}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB 컬렉션 생성: agentic_ai_docs


In [3]:
# 샘플 문서 (에이전틱 AI 관련 지식)
documents = [
    {
        "id": "doc_001",
        "title": "LangChain 소개",
        "content": """LangChain은 대형 언어 모델을 활용한 애플리케이션 개발을 위한 프레임워크입니다.
        도구, 메모리, 에이전트 등의 모듈을 제공하여 복잡한 AI 파이프라인을 쉽게 구축할 수 있습니다.
        LangChain은 OpenAI, Anthropic, HuggingFace 등 다양한 LLM 공급업체를 지원합니다.
        주요 컴포넌트로는 LLM, 프롬프트 템플릿, 체인, 에이전트, 메모리가 있습니다.""",
        "source": "langchain_docs"
    },
    {
        "id": "doc_002",
        "title": "vLLM 아키텍처",
        "content": """vLLM은 PagedAttention 기술을 사용하여 LLM 추론을 고속화하는 서빙 엔진입니다.
        PagedAttention은 운영체제의 가상 메모리 관리 기법을 KV 캐시에 적용한 혁신적인 기술입니다.
        이를 통해 메모리 낭비를 최소화하고 처리량(Throughput)을 크게 향상시킵니다.
        vLLM은 OpenAI API 호환 인터페이스를 제공하여 기존 코드와 쉽게 통합됩니다.""",
        "source": "vllm_blog"
    },
    {
        "id": "doc_003",
        "title": "RAG 시스템 설계",
        "content": """RAG(Retrieval-Augmented Generation)는 LLM의 지식 한계를 극복하기 위한 아키텍처입니다.
        외부 문서를 벡터 데이터베이스에 저장하고, 쿼리와 유사한 문서를 검색하여 LLM에 컨텍스트로 제공합니다.
        주요 단계: 문서 로딩 → 청킹 → 임베딩 → 벡터 DB 저장 → 쿼리 임베딩 → 유사도 검색 → LLM 생성.
        Vector RAG의 한계: 단순 키워드 매칭 위주, 다단계 추론 어려움.""",
        "source": "rag_tutorial"
    },
    {
        "id": "doc_004",
        "title": "Graph RAG 소개",
        "content": """Graph RAG는 Vector RAG의 한계를 극복하기 위해 지식 그래프를 활용합니다.
        엔티티(Entity)와 관계(Relationship)를 그래프로 구성하여 복잡한 다단계 추론을 지원합니다.
        Microsoft Research에서 발표한 GraphRAG는 텍스트에서 지식 그래프를 자동 추출합니다.
        Neo4j + LangChain 조합이 가장 널리 사용되며, Cypher 쿼리로 그래프를 탐색합니다.""",
        "source": "graphrag_paper"
    },
    {
        "id": "doc_005",
        "title": "NeMo Guardrails",
        "content": """NVIDIA NeMo Guardrails는 AI 에이전트의 안전성을 보장하는 오픈소스 프레임워크입니다.
        Colang라는 DSL(Domain Specific Language)로 대화 흐름과 제약 조건을 정의합니다.
        입력 가드레일로 프롬프트 주입을 방지하고, 출력 가드레일로 부적절한 응답을 필터링합니다.
        결정론적 규칙과 확률적 LLM 추론을 결합하여 신뢰할 수 있는 에이전트를 구현합니다.""",
        "source": "nemo_docs"
    }
]

# 문서를 청크로 분할(rag.chunk_document = utils.chunk_text 래퍼) 후 ChromaDB 에 적재
all_chunks = []
all_ids = []
all_metadatas = []

for doc in documents:
    chunks = rag.chunk_document(doc["content"], chunk_size=50, overlap=10)
    for i, chunk in enumerate(chunks):
        chunk_id = f"{doc['id']}_chunk_{i}"
        all_chunks.append(chunk)
        all_ids.append(chunk_id)
        all_metadatas.append({
            "doc_id": doc["id"],
            "title": doc["title"],
            "source": doc["source"],
            "chunk_index": i
        })

collection.add(
    documents=all_chunks,
    ids=all_ids,
    metadatas=all_metadatas
)

print(f"총 {len(all_chunks)}개 청크 저장 완료")
print(f"문서 수: {len(documents)}개")

총 6개 청크 저장 완료
문서 수: 5개


In [4]:
# 벡터 RAG 검색/생성은 agentic_lib.rag.VectorRAG 로 분리했습니다.
#   - 검색: ChromaDB 컬렉션 유사도 질의
#   - 생성: 주입된 llm 으로 답변 생성 후 to_text 로 정규화(<think>/list 처리)
vector_rag = rag.VectorRAG(collection, llm=llm)

# RAG 테스트
test_queries = [
    "vLLM의 PagedAttention이란 무엇인가요?",
    "Graph RAG와 Vector RAG의 차이점은?",
    "NeMo Guardrails에서 Colang의 역할은?",
]

for query in test_queries:
    answer = vector_rag.generate(query)  # 내부에서 to_text 정규화된 문자열 반환
    # 답변을 글자 수로 자르면 표/목록형 응답이 중간에서 끊겨 내용이 사라진다 → 전문을 그대로 출력한다.
    print(f"\n답변: {answer}")
    print("-" * 60)



=== RAG 검색 결과 (쿼리: 'vLLM의 PagedAttention이란 무엇인가요?') ===
  [0.592] [vLLM 아키텍처] vLLM은 PagedAttention 기술을 사용하여 LLM 추론을 고속화하는 서빙 엔진입니다. PagedAttention은 운영체제의 가상 메...
  [0.376] [RAG 시스템 설계] RAG(Retrieval-Augmented Generation)는 LLM의 지식 한계를 극복하기 위한 아키텍처입니다. 외부 문서를 벡터 데이터베...
  [0.329] [Graph RAG 소개] Graph RAG는 Vector RAG의 한계를 극복하기 위해 지식 그래프를 활용합니다. 엔티티(Entity)와 관계(Relationship)를...



답변: vLLM의 PagedAttention은 운영체제의 가상 메모리 관리 기법을 KV 캐시에 적용한 기술로, 메모리 페이징 방식을 통해 KV 캐시의 메모리 낭비를 최소화하고 추론 처리량을 크게 향상시키는 방법입니다.
------------------------------------------------------------

=== RAG 검색 결과 (쿼리: 'Graph RAG와 Vector RAG의 차이점은?') ===
  [0.683] [Graph RAG 소개] Graph RAG는 Vector RAG의 한계를 극복하기 위해 지식 그래프를 활용합니다. 엔티티(Entity)와 관계(Relationship)를...
  [0.664] [RAG 시스템 설계] 생성. Vector RAG의 한계: 단순 키워드 매칭 위주, 다단계 추론 어려움....
  [0.468] [RAG 시스템 설계] RAG(Retrieval-Augmented Generation)는 LLM의 지식 한계를 극복하기 위한 아키텍처입니다. 외부 문서를 벡터 데이터베...

답변: Graph RAG는 엔티티와 관계를 그래프 형태로 구성해 Neo4j+LangChain과 Cypher 쿼리로 다단계 추론을 지원하는 반면, Vector RAG는 문서를 벡터 데이터베이스에 저장하고 유사도 검색(간단한 키워드 매칭 위주)으로 관련 문서를 찾아 LLM에 제공하며, 다단계 추론이 어려운 점이 차이점입니다.
------------------------------------------------------------

=== RAG 검색 결과 (쿼리: 'NeMo Guardrails에서 Colang의 역할은?') ===
  [0.457] [NeMo Guardrails] NVIDIA NeMo Guardrails는 AI 에이전트의 안전성을 보장하는 오픈소스 프레임워크입니다. Colang라는 DSL(Domain Sp...
  [0.213] [LangChain 소개] LangChain은 대형 언어 모델을 활용한 애플리케이션 개발

In [5]:
# 지식 그래프 구현(KnowledgeGraph)은 agentic_lib.rag 로 분리했습니다(Neo4j 없이 개념 학습용).
# 여기서는 AI 기술 생태계를 노드(엔티티)/엣지(관계)로 직접 구성합니다.
kg = rag.KnowledgeGraph()

# 노드 추가
kg.add_node("LangChain", "Framework", description="LLM 애플리케이션 프레임워크", company="LangChain Inc.")
kg.add_node("LangGraph", "Framework", description="에이전트 워크플로우 프레임워크", company="LangChain Inc.")
kg.add_node("vLLM", "Engine", description="고성능 LLM 서빙 엔진", company="Berkeley AI")
kg.add_node("ChromaDB", "Database", description="벡터 데이터베이스", company="Chroma")
kg.add_node("Neo4j", "Database", description="그래프 데이터베이스", company="Neo4j Inc.")
kg.add_node("NeMo", "Framework", description="AI 거버넌스 프레임워크", company="NVIDIA")
kg.add_node("PagedAttention", "Algorithm", description="효율적인 KV 캐시 관리 알고리즘")
kg.add_node("Colang", "Language", description="대화 흐름 정의 DSL")
kg.add_node("Anthropic", "Company", description="AI 안전 연구 기업")
kg.add_node("Claude", "Model", description="Anthropic의 대형 언어 모델")
kg.add_node("MCP", "Protocol", description="Model Context Protocol")

# 엣지 추가
kg.add_edge("LangChain", "includes", "LangGraph")
kg.add_edge("LangChain", "integrates", "vLLM")
kg.add_edge("LangChain", "integrates", "ChromaDB")
kg.add_edge("LangChain", "integrates", "Neo4j")
kg.add_edge("vLLM", "implements", "PagedAttention")
kg.add_edge("NeMo", "uses", "Colang")
kg.add_edge("Anthropic", "develops", "Claude")
kg.add_edge("Anthropic", "created", "MCP")
kg.add_edge("Claude", "supports", "MCP")
kg.add_edge("LangChain", "integrates", "Claude")

print("=== 지식 그래프 구성 완료 ===")
print(f"노드 수: {len(kg.nodes)}")
print(f"엣지 수: {len(kg.edges)}")

print("\n=== LangChain 관련 노드 탐색 ===")
neighbors = kg.query_neighbors("LangChain")
for n in neighbors:
    print(f"  LangChain -[{n['relation']}]-> {n['node']}")

print("\n=== Multi-hop 탐색 (Anthropic → 2홉) ===")
paths = kg.multi_hop_query("Anthropic", max_hops=2)
for path in paths:
    path_str = " -> ".join([f"-[{rel}]-> {node}" for rel, node in path])
    print(f"  Anthropic {path_str}")

=== 지식 그래프 구성 완료 ===
노드 수: 11
엣지 수: 10

=== LangChain 관련 노드 탐색 ===
  LangChain -[includes]-> LangGraph
  LangChain -[integrates]-> vLLM
  LangChain -[integrates]-> ChromaDB
  LangChain -[integrates]-> Neo4j
  LangChain -[integrates]-> Claude

=== Multi-hop 탐색 (Anthropic → 2홉) ===
  Anthropic -[develops]-> Claude
  Anthropic -[created]-> MCP


In [6]:
# Hybrid RAG (Vector + Graph) 는 agentic_lib.rag.HybridRAG 로 분리했습니다.
# 벡터 검색 결과와 그래프 컨텍스트를 합쳐 주입된 llm 으로 답변 생성(to_text 정규화).
hybrid_rag = rag.HybridRAG(vector_rag, kg, llm=llm)


def show(label: str, text: str) -> None:
    """답변을 자르지 않고 라벨·구분선과 함께 출력한다(표 형식 응답이 끊기지 않도록)."""
    print(f"\n[{label}]\n{text}\n" + "-" * 60 + "\n\n")


# (1) 목록형 질문 — 답이 '문서 본문'이 아니라 '엔티티 간 관계'에 있다.
#     샘플 문서 어디에도 "LangChain 이 어떤 DB 와 통합되는지" 는 적혀 있지 않고 그래프에만 있으므로,
#     벡터 단독 답변과 하이브리드 답변을 나란히 찍어 그래프가 무엇을 보태는지 확인한다.
q_list = "LangChain이 지원하는 데이터베이스는 무엇이 있으며, 각각 어떤 용도로 사용하나요?"
show("① Vector RAG 단독", vector_rag.generate(q_list, verbose=False))
show("① Hybrid RAG (Vector + Graph)", hybrid_rag.generate_answer(q_list))

# (2) 다단계(multi-hop) 질문 — Anthropic-[develops]->Claude-[supports]->MCP 와
#     LangChain-[integrates]->Claude 를 그래프가 이어 준다(벡터 유사도로는 닿기 어려운 경로).
q_hop = "Claude는 어떤 회사가 만들었고, LangChain 및 MCP 와는 각각 어떻게 이어지나요?"
show("② Hybrid RAG (multi-hop)", hybrid_rag.generate_answer(q_hop))

# (3) 근거가 없는 질문 — 두 엔티티를 잇는 관계가 그래프에도 문서에도 없다.
#     "모른다"고 답하는지(환각하지 않는지) 확인하는 음성(negative) 사례다.
q_none = "Anthropic과 vLLM의 관계는 무엇인가요?"
show("③ 근거 없음 — 환각하지 않는지 확인", hybrid_rag.generate_answer(q_none))



[① Vector RAG 단독]
LangChain은 벡터 데이터베이스와 Neo4j 그래프 데이터베이스를 지원합니다.  

- **벡터 데이터베이스**: 문서를 임베딩하여 저장하고, 쿼리와 유사한 문서를 검색해 LLM에 컨텍스트를 제공하는 RAG 파이프라인에 사용됩니다.  
- **Neo4j 그래프 데이터베이스**: 엔티티와 관계를 그래프로 저장하여 다단계 추론이 가능한 Graph RAG에서 Cypher 쿼리로 그래프를 탐색하는 데 사용됩니다.
------------------------------------------------------------


=== Hybrid RAG 검색 ===
쿼리: LangChain이 지원하는 데이터베이스는 무엇이 있으며, 각각 어떤 용도로 사용하나요?
발견된 엔티티: ['LangChain']

[① Hybrid RAG (Vector + Graph)]
LangChain은 다양한 외부 저장소와 연동하여 데이터를 효율적으로 관리하고 검색할 수 있도록 설계되어 있습니다. 제공된 컨텍스트를 바탕으로 LangChain이 직접 통합하거나 지원한다고 언급된 데이터베이스와 그 주요 용도는 다음과 같습니다.

| 데이터베이스 | LangChain과의 연동 방식 | 주요 용도 |
|--------------|----------------------|----------|
| **ChromaDB** | `LangChain -[integrates]-> ChromaDB` (벡터 데이터베이스) | - 문서나 텍스트 청크를 임베딩 벡터로 변환하여 저장 <br> - 유사도 검색(코사인 유사도, 내적 등)을 통해 쿼리와 가장 관련된 문서를 빠르게 찾아 LLM에 컨텍스트로 제공 <br> - Vector RAG 파이프라인에서 기본적인 검색 엔진 역할 수행 |
| **Neo4j** | `LangChain -[integrates]-> Neo4j` (그래프 데이터베이스) | - 엔티티(Entity)와 관계(Relationship)를 노드와 엣지로 모델링하여 지식 그

---
## 6. 에이전트 메모리 시스템

앞의 RAG 는 "**문서** 에서 찾기"였습니다. 이제 에이전트가 **겪은 일과 아는 사실** 을 스스로 들고 다니게 합니다.
구현은 [`agentic_lib/agent_memory.py`](agentic_lib/agent_memory.py) 로 분리했고, 노트북에서는 **쓰면서 관찰** 합니다.

### 4계층 — 무엇을, 얼마나 오래, 어떻게 찾는가

| 계층 | 클래스 | 저장 위치 | 수명 | 검색 방식 | 넣는 주체 |
|---|---|---|---|---|---|
| **작업**(Working) | `AgentMemorySystem.working_memory` | dict | 지금 이 요청 | 직접 참조 | 시스템 |
| **단기**(Short-term) | `ShortTermMemory` | `deque(maxlen)` | 최근 N턴 | 통째로 전달 | 자동 — 모든 대화 |
| **일화**(Episodic) | `EpisodicMemory` | ChromaDB | 영구 | 임베딩 유사도 | 자동 — 단, `important=True` 만 |
| **의미**(Semantic) | `SemanticMemory` | dict | 영구 | 부분 문자열 | **사람이 `learn()`** |

### 왜 계층을 나누는가

한 계층으로 전부 처리하려 들면 반드시 한쪽이 무너집니다.

- **전부 단기로** — 잊지 않으니 프롬프트가 줄지 않고, 대화 길이가 그대로 토큰 비용이 됩니다.
  `deque(maxlen)` 이 "잊는" 것은 버그가 아니라 **비용을 상수로 묶는 설계** 입니다.
- **전부 일화(벡터)로** — 모든 발화를 임베딩하니 비용도 잡음도 함께 늘고,
  방금 한 말조차 유사도 검색으로 되찾아야 합니다.
- **의미 기억은 저절로 안 쌓입니다** — 확정된 사실은 대화 요약이 아니라 누군가 **넣어 줘야** 하는 것이고,
  그래서 신뢰도(`confidence`)와 출처를 함께 들고 다닙니다.

에이전트가 쓰는 API 는 결국 두 개뿐입니다 — `perceive()`/`respond()` 로 흘려 넣고,
`get_relevant_context()` 로 지금 필요한 기억만 하나의 문자열로 받습니다.
이 경계 덕분에 일화 기억을 Chroma → Qdrant 로 바꿔도 에이전트 코드는 그대로입니다.


In [7]:
# 4계층 메모리 구현은 agentic_lib.agent_memory 로 분리했습니다(셋업 셀에서 am 으로 import).
# 여기서는 조립하고, 계층별로 무엇이 어떻게 남는지 관찰합니다.

# 일화 기억 전용 컬렉션 — 지식 문서(agentic_ai_docs)와 '겪은 일'을 섞지 않는다
episode_collection = chroma_client.get_or_create_collection(
    "agent_episodes", embedding_function=ef
)
memory_system = am.AgentMemorySystem(episode_collection)

# ① 의미 기억 — 사실은 대화에서 저절로 쌓이지 않으므로 직접 학습시킨다
for fact, conf in [
    ("LangGraph는 순환 그래프 기반 에이전트 프레임워크다", 0.95),
    ("vLLM은 PagedAttention으로 처리량을 향상시킨다", 0.90),
    ("Neo4j는 Cypher 쿼리 언어를 사용하는 그래프 DB다", 0.95),
]:
    memory_system.semantic.learn(fact, confidence=conf)

# ② 대화 — perceive/respond 만 부르면 어느 계층에 남길지는 메모리 시스템이 정한다.
#    important=True 인 응답만 일화 기억(벡터 DB)으로 넘어간다.
print("=== 대화 시뮬레이션 (important=True 인 것만 장기 기억으로) ===")
conversations = [
    ("LangGraph가 무엇인지 설명해줘",
     "LangGraph는 상태 관리와 루프를 지원하는 에이전트 워크플로우 프레임워크입니다.", True),
    ("Neo4j와 어떻게 연동하나요?",
     "Neo4j는 Python neo4j 라이브러리와 LangChain Neo4jGraph를 통해 연동합니다.", True),
    ("고마워요", "천만에요!", False),   # 잡담 — 단기에만 남고 벡터 DB 로는 가지 않는다
]

for user_msg, bot_response, important in conversations:
    memory_system.perceive(user_msg)
    memory_system.respond(bot_response, important=important)
    print(f"[사용자] {user_msg}")
    print(f"[에이전트] {bot_response}  (장기 저장: {important})\n")

# ③ 계층별 보유량 — 단기에는 3턴이 모두, 일화에는 중요한 2건만 남는다
print("=== 계층별 상태 ===")
for key, value in memory_system.stats().items():
    print(f"  {key}: {value}")

# ④ 같은 질문으로 두 계층을 조회해 성격 차이를 대비시킨다
query = "그래프 데이터베이스 연동"
print(f"\n=== '{query}' 로 각 계층 조회 ===")

facts = memory_system.semantic.search(query)
print(f"  의미 기억(부분 문자열 매칭): {len(facts)}건"
      "  ← 사실에는 '그래프 DB'라고 적혀 있어 '데이터베이스'라는 표현으로는 못 찾는다")

recalled = memory_system.episodic.recall(query, n_results=2)
print(f"  일화 기억(임베딩 유사도): {len(recalled)}건  ← 표현이 달라도 '후보'는 찾아낸다")
for rank, episode in enumerate(recalled, start=1):
    print(f"      {rank}위 (거리 {episode['distance']:.3f}) {episode['content'][:55]}...")

# 순위는 임베딩 모델이 정한다 — 'Neo4j 연동' 대화가 1위가 아닐 수도 있다.
# 찾히느냐(계층의 성격)와 잘 찾느냐(모델 품질)는 별개 문제다(M04_1 의 한국어 모델 비교 참고).
print("  ※ 순위가 기대와 다르면 메모리 구조가 아니라 임베딩 모델을 의심한다")

# ⑤ 에이전트가 실제로 받는 것 — 세 계층을 합친 하나의 컨텍스트 문자열
print("\n=== get_relevant_context() 가 돌려주는 컨텍스트 ===")
print(memory_system.get_relevant_context("LangGraph와 그래프 DB 연동"))


=== 대화 시뮬레이션 (important=True 인 것만 장기 기억으로) ===
에피소드 저장: [episode_0001] Q: LangGraph가 무엇인지 설명해줘 A: LangGraph는 상태 관리와 루프를 지원하는 에이전트 워...
[사용자] LangGraph가 무엇인지 설명해줘
[에이전트] LangGraph는 상태 관리와 루프를 지원하는 에이전트 워크플로우 프레임워크입니다.  (장기 저장: True)

에피소드 저장: [episode_0002] Q: Neo4j와 어떻게 연동하나요? A: Neo4j는 Python neo4j 라이브러리와 LangChain...
[사용자] Neo4j와 어떻게 연동하나요?
[에이전트] Neo4j는 Python neo4j 라이브러리와 LangChain Neo4jGraph를 통해 연동합니다.  (장기 저장: True)

[사용자] 고마워요
[에이전트] 천만에요!  (장기 저장: False)

=== 계층별 상태 ===
  단기(메시지): 6
  단기(한도): 20
  일화(저장 건수): 2
  의미(사실 수): 3
  작업(키): ['last_input']

=== '그래프 데이터베이스 연동' 로 각 계층 조회 ===
  의미 기억(부분 문자열 매칭): 0건  ← 사실에는 '그래프 DB'라고 적혀 있어 '데이터베이스'라는 표현으로는 못 찾는다
  일화 기억(임베딩 유사도): 2건  ← 표현이 달라도 '후보'는 찾아낸다
      1위 (거리 0.654) Q: LangGraph가 무엇인지 설명해줘 A: LangGraph는 상태 관리와 루프를 지원하는 에...
      2위 (거리 0.845) Q: Neo4j와 어떻게 연동하나요? A: Neo4j는 Python neo4j 라이브러리와 Lang...
  ※ 순위가 기대와 다르면 메모리 구조가 아니라 임베딩 모델을 의심한다

=== get_relevant_context() 가 돌려주는 컨텍스트 ===
[최근 대화]
[user] LangGraph가 무엇인지 설명해줘
[assistant

---
## 7. Deep-Knowledge Agent 최종 구현

지금까지 만든 것을 하나로 잇습니다 — **Hybrid RAG(벡터 + 그래프) + 4계층 메모리 + 추론 체인**.
구현은 [`agentic_lib/agent_memory.py`](agentic_lib/agent_memory.py) 의 `DeepKnowledgeAgent` 이고,
노트북에서는 **조립하고 파이프라인이 어떻게 도는지 관찰** 합니다.

### 질문 하나가 지나가는 네 단계

| 단계 | 메서드 | 하는 일 | 여기서 틀리면 보이는 증상 |
|---|---|---|---|
| 사고 | `think()` | 어떤 소스를 볼지 전략을 한 문장으로 기록 | — |
| 검색 | `retrieve()` | Hybrid RAG 검색 + 메모리 회상을 한 번에 수집 | 엔티티 0개, 근거 없는 답변 |
| 추론 | `reason()` | 모인 근거가 무엇인지 요약해 기록 | 근거와 답이 서로 어긋남 |
| 생성 | `generate_answer()` | 근거 + 추론 메모를 LLM 에 넘겨 최종 답 생성 | 근거는 맞는데 종합이 틀림 |

**단계를 쪼개는 이유는 속도가 아니라 관찰 가능성입니다.** 답이 틀렸을 때
검색이 틀린 건지(`retrieve`) 종합이 틀린 건지(`generate`) 구분되지 않으면 고칠 수가 없습니다.
그래서 모든 단계가 `reasoning_chain` 에 남고, 반환값에는 **`sources`**(벡터 몇 건 · 그래프 어떤 엔티티)가
답변과 함께 옵니다. 3편에서 강조한 "체인 중간을 볼 수 없으면 고칠 수도 없다"와 같은 원칙입니다.

> 최종 답변은 `important=True` 로 **일화 기억에 자동 저장** 됩니다.
> 그래서 두 번째 질문부터는 첫 질문의 답이 기억으로 되돌아와 컨텍스트에 섞입니다 — 아래에서 직접 확인합니다.


In [8]:
# DeepKnowledgeAgent 구현은 agentic_lib.agent_memory 로 분리했습니다(여기서는 조립·관찰만).
# 세 부품을 주입하기만 하면 된다 — 검색기 / 메모리 / LLM
dk_agent = am.DeepKnowledgeAgent(
    hybrid_rag=hybrid_rag,        # 5절: 벡터 + 그래프 결합 검색
    memory_system=memory_system,  # 6절: 4계층 메모리
    llm=llm,                      # .env 의 LLM_PROVIDER 그대로
)

# ① 첫 질문 — 이 주제에 대한 기억은 아직 없다. 사고 → 검색 → 추론 → 생성이 순서대로 출력된다.
result = dk_agent.generate_answer(
    "LangChain을 vLLM과 함께 사용할 때의 장점은 무엇이며, Graph RAG를 어떻게 통합할 수 있나요?"
)

# ② 반환값에는 답변만이 아니라 '무엇을 근거로 했는지'가 함께 온다
print("\n=== 근거(sources) ===")
for source in result["sources"]:
    print(" ", source)
print(f"추론 체인 단계 수: {len(result['reasoning_chain'])}")

# ③ 이어지는 질문 — ①의 답이 일화 기억에 저장됐으므로 '방금 이야기한'을 되살릴 수 있다
follow_up = dk_agent.generate_answer("방금 이야기한 통합 방식에서 Neo4j는 어떤 역할을 하나요?")

# ④ 대화를 거치며 기억이 실제로 쌓였는지 확인
print("\n=== 대화 후 계층별 상태 ===")
for key, value in memory_system.stats().items():
    print(f"  {key}: {value}")



질문: LangChain을 vLLM과 함께 사용할 때의 장점은 무엇이며, Graph RAG를 어떻게 통합할 수 있나요?

[사고] 'LangChain을 vLLM과 함께 사용할 때의 장점은 무엇이며, Graph RAG를 어떻게 통합할 수 있나요?'를 이해하기 위해 지식 그래프와 벡터 DB를 검색해야 한다
[추론] 지식 그래프에서 발견된 엔티티: ['LangChain', 'vLLM']
벡터 검색 결과: 3개
기억에서 관련 컨텍스트: 있음

이 정보들을 종합하여 'LangChain을 vLLM과 함께 사용할 때의 장점은 무엇이며, Graph RAG를 어떻게 통합할 수 있나요?'에 답할 수 있다.
에피소드 저장: [episode_0003] Q: LangChain을 vLLM과 함께 사용할 때의 장점은 무엇이며, Graph RAG를 어떻게 통합할 수...

[최종 답변] **LangChain + vLLM을 함께 사용할 때의 주요 장점**

| 장점 | 설명 | 관련 내용 |
|------|------|-----------|
| **추론 속도·처리량 향상** | vLLM은 **PagedAttention** 기술을 적용해 KV 캐시를 페이지 단위로 관리함으로써 메모리 낭비를 최소화하고, 동일 하드웨어에서 **Throughput(초당 토큰 수)** 를 크게 끌어올립니다. | 벡터 검색 결과: *vLLM 아키텍처* – PagedAttention 설명 |
| **메모리 효율성** | 페이지 기반 캐시 할당으로 긴 컨텍스트(수만 토큰)도 비교적 적은 GPU 메모리로 처리 가능 → 대형 모델이나 배치 크기를 늘릴 수 있음. |同上 |
| **OpenAI‑호환 인터페이스** | vLLM은 OpenAI API와 동일한 엔드포인트(`/v1/completions`, `/v1/chat/completions`)를 제공하므로, LangChain의 `ChatOpenAI` 또는 `OpenAI` 래퍼를 그대로 교체만 하면 바로 사용할 수 있습니다. | 벡터 검색 결과: *vLLM 아키텍처* – Ope

---
## 8. 정리 및 다음 모듈 예고

### 핵심 정리
1. **Vector RAG:** 텍스트 → 임베딩 → 벡터 DB → 유사도 검색 → LLM 생성
2. **Graph RAG:** 엔티티/관계 그래프로 다단계 추론 지원
3. **Hybrid RAG:** Vector + Graph 결합으로 최고 성능 달성
4. **에이전트 메모리:**
   - 단기: 슬라이딩 윈도우 대화 버퍼
   - 에피소딕: 중요 사건 벡터 저장
   - 시맨틱: 사실/지식 명시적 저장
5. **Multi-hop 추론:** 그래프 경로 탐색으로 간접 관계 발견

### 다음 편: [`M04_5_memory.ipynb`](M04_5_memory.ipynb)

여기서는 메모리를 **직접 만들어** 구조를 이해했습니다. 5편은 같은 개념을
**LangGraph 표준 부품**으로 다시 세웁니다 — 실무 코드는 대부분 이 형태입니다.

- `MessagesState` + `MemorySaver` — 스레드(`thread_id`)별 대화 자동 누적
- `SqliteSaver` — 프로세스를 재시작해도 남는 **영속** 기억
- LangGraph `Store` — 스레드를 넘나드는 사용자 프로필/선호 저장
- 요약 메모리 · 메모리 전략 선택 가이드(비용 vs 성능)

---
### 참고 자료
- Microsoft GraphRAG: https://microsoft.github.io/graphrag/
- ChromaDB 문서: https://docs.trychroma.com
- Neo4j + LangChain: https://python.langchain.com/docs/integrations/graphs/neo4j_cypher
- Generative Agents 논문: https://arxiv.org/abs/2304.03442